# 챔피언 추천

## Imports

In [1]:
import pandas as pd
#import json
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import random_split
from torch.utils.data import DataLoader, TensorDataset

# matplotlib 한글 폰트 설정
if os.name == 'nt':
    plt.rc('font', family='Malgun Gothic')
elif os.name == 'posix':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

# cuda 자동선택
device = 'cuda' if torch.cuda.is_available() else 'cpu'
#device = 'cpu'

### 승패예측모델

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, 200)
        self.dropout1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(200, 100)
        self.dropout2 = nn.Dropout(0.4)
        self.fc3 = nn.Linear(100, 1)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = F.sigmoid(self.fc3(x))
        return x

In [3]:
# train, test dataset

# read dataset
df = pd.read_csv('WinPredDataset_v2.csv')
df = df.iloc[:-1].astype('float32')

# target : 'win'
# split dataset
X = df.drop(columns=['win'])
y = df['win']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# torch data
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

# dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# dataloader
train_loader = DataLoader(train_dataset, batch_size=10000, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=10000, shuffle=False)

In [4]:
# load model
model = MLP(X_train.shape[1]).to(device)
model.load_state_dict(torch.load('model.pth'))
model.eval()

MLP(
  (fc1): Linear(in_features=190, out_features=200, bias=True)
  (dropout1): Dropout(p=0.4, inplace=False)
  (fc2): Linear(in_features=200, out_features=100, bias=True)
  (dropout2): Dropout(p=0.4, inplace=False)
  (fc3): Linear(in_features=100, out_features=1, bias=True)
)

### 챔피언 정보

In [13]:
# read chamnpion_tag
champ_tag = pd.read_csv('champ_tag.csv', index_col=1)
champ_tag.drop(columns='antiAP', inplace=True)
champ_tag.drop(columns='antiAD', inplace=True)
champ_tag.drop(columns='championId', inplace=True)

champ_tag.fillna(0, inplace=True)

# 3번 컬럼부터 int로 변경
for col in champ_tag.columns[2:]:
    champ_tag[col] = champ_tag[col].astype('int')


## 챔피언 추천 과정
1. 유저 풀 지정
2. 밴픽상황 적용
3. 유저 풀에 있는 챔피언을 하나씩 모델에 적용하여 모델의 기대승률 도출
4. 도출한 기대승률을 기반으로 오름차순 정렬

### 유저 풀 지정

In [14]:
### 챔피언 추천 과정
# 1. 유저 풀 지정
# 2. 밴픽상황 적용
# 3. 유저 풀에 있는 챔피언을 하나씩 모델에 적용하여 모델의 기대승률 도출
# 4. 도출한 기대승률을 기반으로 오름차순 정렬

# Simple Example
# 1. 유저 풀 지정
user_pool = ["누누와 윌럼프", "람머스", "워윅", "에코", "릴리아"]


### 밴픽상황 적용

In [15]:
# 2. 밴픽상황 적용
# 유저 블루 4픽, 유저팀: 이렐리아, 럭스, 세나 픽 완료
# 레드 3픽까지 픽 리 신, 아칼리, 라칸

b_team = ["이렐리아", "럭스", "세나"]
r_team = ["리 신", "아칼리", "라칸"]

### 유저 풀에 있는 챔피언을 하나씩 모델에 적용하여 챔피언별 기대승률 도출

In [ ]:
# 3. 유저 풀에 있는 챔피언을 하나씩 모델에 적용하여 모델의 기대승률 도출
tag_lineup_dict = {}
for tag_name in champ_tag.columns.to_list():
    for i in range(10):
        tag_lineup_dict[f'{i}_{tag_name}'] = []

# 블루팀
for b, champ in enumerate(b_team):
    for tag_name in champ_tag.columns.to_list():
        tag_lineup_dict[f'{b}_{tag_name}'] = champ_tag.loc[champ, tag_name]
# 레드팀
for r, champ in enumerate(r_team):
    for tag_name in champ_tag.columns.to_list():
        tag_lineup_dict[f'{r+5}_{tag_name}'] = champ_tag.loc[champ, tag_name]
r = r+5


print(tag_lineup_dict)
# 기대승률 도출
for champ in user_pool:
    for tag_name in champ_tag.columns.to_list():
        
    
# conver to numpy array
lineup_array = np.array(list(tag_lineup_dict.values()))
# tensor
print(lineup_array.shape)
lineup = torch.Tensor(lineup_array, dtype='float32', device=device)

expected_winrate = {}
# predict
with torch.no_grad():
    pred = model(lineup)
expected_winrate[champ] = pred.item()
print(f'{champ} : {pred.item()}')

{'0_전사': 1.0, '1_전사': 0.0, '2_전사': 0.0, '3_전사': 0, '4_전사': 0, '5_전사': 1.0, '6_전사': 0.0, '7_전사': 0.0, '8_전사': 0, '9_전사': 0, '0_암살자': 1.0, '1_암살자': 0.0, '2_암살자': 0.0, '3_암살자': 0, '4_암살자': 0, '5_암살자': 1.0, '6_암살자': 1.0, '7_암살자': 0.0, '8_암살자': 0, '9_암살자': 0, '0_마법사': 0, '1_마법사': 1, '2_마법사': 0, '3_마법사': 0, '4_마법사': 0, '5_마법사': 0, '6_마법사': 0, '7_마법사': 0, '8_마법사': 0, '9_마법사': 0, '0_원거리 공격': 0, '1_원거리 공격': 0, '2_원거리 공격': 1, '3_원거리 공격': 0, '4_원거리 공격': 0, '5_원거리 공격': 0, '6_원거리 공격': 0, '7_원거리 공격': 0, '8_원거리 공격': 0, '9_원거리 공격': 0, '0_지원가': 0, '1_지원가': 1, '2_지원가': 1, '3_지원가': 0, '4_지원가': 0, '5_지원가': 0, '6_지원가': 0, '7_지원가': 1, '8_지원가': 0, '9_지원가': 0, '0_탱커': 0, '1_탱커': 0, '2_탱커': 0, '3_탱커': 0, '4_탱커': 0, '5_탱커': 0, '6_탱커': 0, '7_탱커': 0, '8_탱커': 0, '9_탱커': 0, '0_AP': 0, '1_AP': 1, '2_AP': 0, '3_AP': 0, '4_AP': 0, '5_AP': 0, '6_AP': 1, '7_AP': 0, '8_AP': 0, '9_AP': 0, '0_AD': 1, '1_AD': 0, '2_AD': 0, '3_AD': 0, '4_AD': 0, '5_AD': 1, '6_AD': 0, '7_AD': 0, '8_AD': 0, '9_AD': 0, '0_CC': 0, '1_CC': 0, '2_

TypeError: new() received an invalid combination of arguments - got (numpy.ndarray, device=str, dtype=str), but expected one of:
 * (*, torch.device device)
 * (torch.Storage storage)
 * (Tensor other)
 * (tuple of ints size, *, torch.device device)
      didn't match because some of the keywords were incorrect: dtype
 * (object data, *, torch.device device)
      didn't match because some of the keywords were incorrect: dtype
